# H3 Toybox｜Colab 网页版

把想象丢进玩具箱，摇一摇就变成视频 🧸 ComfyUI 只在背景当引擎，**不用开 ComfyUI 界面**，所有操作都在 H3 Toybox 网页里。

运行环境选 **L4 24 GB** 或 **A100 40/80 GB**。选择「运行时 → 全部运行」，在第 ③ 格的输出区操作 Toybox。

| 步骤 | 做什么 |
|---|---|
| ① 检查并安装 | 更新 ComfyUI 主分支、安装依赖与视频节点，从 GitHub 拉取 Toybox |
| ② 下载模型 | 视频约 50 GB，P-H3-Image 另加约 10 GB，H3-Image 另加约 22.15 GB。下载前按所选模型检查磁盘 |
| ③ 启动并显示网页 | 后台启动 ComfyUI 和 H3 Toybox，在本单元下方内嵌网页；作品默认存到 Google Drive 的「H3Toybox」文件夹 |
| ③-b | 备用 Cloudflare 链接，默认关闭，可按需手动开启 |

**网页里能做的事**
- 🖼 **图片**（选用）：文生图，或拿最多 8 张图做编辑（换表情、换衣服、换风格、把人放进场景）。用 [P-H3-Image](https://huggingface.co/PrunaAI/P-H3-Image-Comfy) 挂在同一颗 H3 模型上，只多约 10 GB
- **H3-Image（4K 生图）**：1K、2K、4K、自定义宽高，支持最多 16 张参考图。生成后可把原图发到视频首帧
- 🤖 **短片 Agent**：一句话规划成多段短片（要先在 ⚙ 设置填 Gemini 密钥）
- 🧸 **参考生成**：图片 9 张／视频 3 支／声音 3 段当参考，提示词里用 `@图片1` 标记
- 🎞 **首尾帧**：只放首帧、只放尾帧、或两张都放
- ✏️ **纯文本**：只靠一段描述
- 🧵 **延伸视频**：参考视频勾「延伸这支视频」，新片段会从它的最后一格接下去，并自动接回原片
- ✨ **提示词改写**：把中文描述扩写成 H3 规格的英文提示词（改写结果默认收起来）。默认用 H3 官方的 9B 改写模型；也可以在 ⚙ 设置改用 **Gemini** 或 **OpenAI 兼容端点**
- ⚡ **快速模式**（默认开）：8 步出片；关掉是 12 步
- 📁 **素材库文件夹**：把图片、视频、声音拖进文件夹整理
- 每支视频都会显示目前做到哪一步、还要多久

**Hugging Face 令牌（可选）**：可以在 Colab 左侧「密钥」添加只读 `HF_TOKEN`。

模型：`Minimax-h3_Singularity_x_10Eros-Max_PRISM`（int8）＋官方 Qwen3-VL 文本编码器与 VAE；LoRA、语义桥、潜空间放大器与改写模型镜像在 [dseditor/minimax-h3-studio-assets](https://huggingface.co/dseditor/minimax-h3-studio-assets)（各档来源见该页）。授权依照各来源模型。

L4 的 4K 和多图任务取决于可用显存与主机内存。可先用 1K 或 2K 确认运行环境。

In [ ]:
#@title ① 检查环境＋安装 ComfyUI、节点与 H3 Toybox（GPU 或驱动不符就停下，不会安装）
import os, shutil, subprocess, sys, torch
from pathlib import Path
from google.colab import userdata

if not torch.cuda.is_available():
    raise RuntimeError('没有 GPU：请到「运行阶段 → 变更运行阶段类型」选 L4 GPU，再重跑这格')
cap = torch.cuda.get_device_capability(0)
drv = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', torch.cuda.get_device_name(0), '| capability', cap, '| VRAM', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('驱动:', drv, '| 可用磁盘', round(shutil.disk_usage('/content').free / 2**30, 1), 'GiB')
if cap < (8, 0):
    raise RuntimeError(f'这张卡（capability {cap}）跑不动 int8 模型，没有安装任何东西。请换 L4（或 A100）再重跑。')
if int(drv.split('.')[0]) < 580:
    raise RuntimeError(f'驱动 {drv} 不支持 CUDA 13（需要 580 以上），没有安装任何东西。请换一台运行阶段（L4）再试。')
print('✅ 环境检查通过，开始安装')
try:
    hf_token = (userdata.get('HF_TOKEN') or '').strip()
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    hf_token = ''
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print('已读取 HF_TOKEN')
else:
    print('未设置 HF_TOKEN，使用匿名下载')

def clone(url, dest, *, branch=None):
    dest = Path(dest)
    if not dest.exists():
        cmd = ['git', 'clone', '--depth', '1']
        if branch:
            cmd += ['--branch', branch]
        cmd += [url, str(dest)]
    else:
        git = ['git', '-C', str(dest)]
        remote = subprocess.check_output(git + ['remote', 'get-url', 'origin'], text=True).strip()
        current = subprocess.check_output(git + ['branch', '--show-current'], text=True).strip()
        if remote.removesuffix('.git') != url.removesuffix('.git') or (branch and current != branch):
            raise RuntimeError(f'{dest} 已检出其他仓库或分支，请在新的 Colab 运行时重试')
        cmd = git + ['pull', '--ff-only']
    subprocess.run(cmd, env=dict(os.environ, GIT_TERMINAL_PROMPT='0'), check=True)

source = Path('/content/minimax-h3-toybox')
clone('https://github.com/Alex-955/minimax-h3-colab.git', source, branch='main')
print('已读取 Toybox 源码')
APP = source / 'toybox'
if not (APP / 'server.py').is_file() or not (source / 'workflows/minimax_h3_image_api.json').is_file():
    raise RuntimeError('该分支缺少改造版 Toybox 或 H3-Image 工作流，请检查是否已推送代码')

COMFY = Path('/content/ComfyUI')
clone('https://github.com/Comfy-Org/ComfyUI.git', COMFY, branch='master')
torch_cuda = subprocess.check_output([sys.executable, '-c', 'import torch; print(torch.version.cuda)'], text=True).strip()
if not torch_cuda.startswith('13'):
    print('安装 CUDA 13 版 PyTorch（int8 ConvRot 权重需要）…')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torch', 'torchvision', 'torchaudio',
                    '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt'), 'huggingface_hub', 'hf_xet'], check=True)
# 工作流用到的三个节点组（都不会动到 PyTorch）
NODES = {
    'minimax-h3-audio-t8': 'https://github.com/T8mars/comfyui-minimax-h3-audio-T8',        # 语义桥 Semantic Bridge
    'comfyui-SelfLift': 'https://github.com/facok/comfyui-SelfLift',                      # 低分辨率→潜空间放大→高分辨率
    'ComfyUI-H3-Motion-Context': 'https://github.com/NikoDemon80/ComfyUI-H3-Motion-Context',  # 延伸视频（气锁）
}
for name, url in NODES.items():
    clone(url, COMFY / 'custom_nodes' / name)
# 导演 Skill 只是文本档：放在 custom_nodes 外面，不加载它的节点
clone('https://github.com/T8mars/comfyui-minimax-h3-prompt-enhancer-T8', '/content/t8-enhancer')
print('ffmpeg:', shutil.which('ffmpeg') or '没有（延伸视频会失败）')
subprocess.run([sys.executable, '-c', 'import torch; assert torch.cuda.is_available() and torch.version.cuda.startswith("13"), (torch.__version__, torch.version.cuda); print("PyTorch", torch.__version__, "CUDA", torch.version.cuda)'],
               check=True)
print('✅ 安装完成')


In [ ]:
#@title ② 下载视频和生图权重
# 默认直接从 Hugging Face 下载（最快）。HF 令牌已在 ① 读入。
# 只有在 Hugging Face 撞到流量上限时，才勾选 USE_DRIVE_CACHE：
#   勾选后会从「我的Google Drive/H3_models」拷贝（比 HF 慢），Google Drive没有的文件才去 HF 抓并顺便存一份。
USE_DRIVE_CACHE = False  #@param {type:"boolean"}
#@markdown 不勾就少下载 7 GB：网页的本地改写会关掉，可以到 ⚙ 设置改用 Gemini 或 OpenAI 端点改写。
INCLUDE_REWRITER = True  #@param {type:"boolean"}
#@markdown 🖼 加装生图（多约 10 GB）：网页的「图片」分页可以文生图、拿最多 8 张图做编辑。挂在同一颗 H3 模型上。
INSTALL_IMAGE = True  #@param {type:"boolean"}
#@markdown H3-Image 独立 4K 生图权重，约 22.15 GB，复用视频的 Qwen3-VL 编码器。
INSTALL_H3_IMAGE = True  #@param {type:"boolean"}
from huggingface_hub import HfApi, hf_hub_download
import shutil, time
from collections import defaultdict
CACHE = None
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = Path('/content/drive/MyDrive/H3_models'); CACHE.mkdir(parents=True, exist_ok=True)
ASSETS = 'dseditor/minimax-h3-studio-assets'
H3_IMAGE_REPO = 'Comfy-Org/MiniMax-H3-Image'
H3_IMAGE_FILES = ('diffusion_models/minimax_h3_image_pruned_int8_convrot.safetensors', 'vae/minimax_h3_image_vae_fp16.safetensors')
SHARED_IMAGE_ENCODER = 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
files = [  # (repo, 文件, 放到 models/ 底下的哪里)
    ('OliviaRossi/Minimax-h3_Singularity-x-10Eros-Max_PRISM', 'Minimax-h3_Singularity_x_10Eros-Max_PRISM_int8.safetensors', 'diffusion_models'),
    ('Comfy-Org/MiniMax-H3', SHARED_IMAGE_ENCODER, 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_int8_convrot.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
    # pika flex Ref2VA（2026-11-04 取代 dmad）：视频与 32×32 TTS 共用这一颗
    ('Comfy-Org/MiniMax-H3', 'loras/pika_h3_flex_ref2va_v1_bf16.safetensors', 'loras'),
] + [(ASSETS, f, str(Path(f).parent)) for f in [
    'loras/minimax_h3_turbo_v4_step600_ema_DasiwaREF2VAHybridV1_0_curveproj1025_compat_v001.safetensors',
    'loras/MysticXXX_MMH3-V2.safetensors',
    'loras/H3_Combat_V2.safetensors',
    'loras/Motion_Repair.safetensors',
    'loras/H3_SPEED_slider.safetensors',
    'semantic_bridge/t8_compat/BUNNY_H3_ActionLogic_Bridge_V1_T8_Compat.safetensors',
    'latent_upscale_models/minimax_h3_latent_upscaler_3d_fp16_V2.safetensors',
] + (['text_encoders/minimax_h3_rewriter_ref2va_qwen3.5_9b_w4a8.safetensors'] if INCLUDE_REWRITER else [])] + ([
    ('PrunaAI/P-H3-Image-Comfy', 'loras/p-h3-image_lora_bf16.safetensors', 'loras'),
    ('PrunaAI/P-H3-Image-Comfy', 'text_encoders/p-h3-image_text_encoder_int8_convrot.safetensors', 'text_encoders'),
    ('PrunaAI/P-H3-Image-Comfy', 'vae/p-h3-image_vae_int8_convrot.safetensors', 'vae'),
] if INSTALL_IMAGE else []) + [
    (H3_IMAGE_REPO, f, str(Path(f).parent)) for f in H3_IMAGE_FILES if INSTALL_H3_IMAGE
]

# 按清单的实际文件大小检查磁盘和缓存。只有完整文件才会进入模型目录。
requested = defaultdict(list)
for repo, name, _ in files:
    requested[repo].append(name)
api = HfApi()
sizes = {(repo, info.path): info.size for repo, names in requested.items()
         for info in api.get_paths_info(repo_id=repo, paths=names)}
missing = [(repo, name) for repo, name, _ in files if not sizes.get((repo, name))]
if missing:
    raise RuntimeError(f'Hugging Face 找不到以下权重：{missing}')

def complete(path, size):
    return path is not None and path.is_file() and path.stat().st_size == size

remaining = sum(sizes[repo, name] for repo, name, folder in files
                if not complete(COMFY / 'models' / folder / Path(name).name, sizes[repo, name]))
free = shutil.disk_usage('/content').free
if remaining and free < remaining + 3 * 2**30:
    raise RuntimeError(f'仍需下载或复制 {remaining / 2**30:.1f} GiB 权重，可用 {free / 2**30:.1f} GiB。'
                       '请释放空间，或取消 INSTALL_IMAGE / INCLUDE_REWRITER 后重试。')

def copy_model(src, dest):
    partial = dest.with_suffix(dest.suffix + '.part')
    try:
        shutil.copyfile(src, partial)
        partial.replace(dest)
    finally:
        partial.unlink(missing_ok=True)

t0 = time.time()
for repo, name, folder in files:
    target = COMFY / 'models' / folder / Path(name).name
    target.parent.mkdir(parents=True, exist_ok=True)
    cached = CACHE / Path(name).name if CACHE else None
    size = sizes[repo, name]
    if not complete(target, size):
        target.unlink(missing_ok=True)
        if complete(cached, size):
            print('从 Google Drive 复制', cached.name)
            copy_model(cached, target)
        else:
            got = Path(hf_hub_download(repo_id=repo, filename=name, local_dir='/content/hf'))
            if not complete(got, size):
                raise RuntimeError(f'权重大小与 Hugging Face 清单不符：{name}')
            got.replace(target)
    if cached and not complete(cached, size):
        print('存一份到 Google Drive', cached.name)
        copy_model(target, cached)
    print(f'{folder:<28} {target.name[:60]:<60} {target.stat().st_size / 2**30:5.1f} GiB')
# 官方提示词写作指南（改写模型照它写）
GUIDES = APP / 'guides'; GUIDES.mkdir(exist_ok=True)
for g in ('VIDEO_PROMPT_WRITING_GUIDE_ref_en.md', 'VIDEO_PROMPT_WRITING_GUIDE_base_en.md'):
    got = Path(hf_hub_download(repo_id='MiniMaxAI/MiniMax-H3', filename='docs/' + g, local_dir='/content/hf'))
    shutil.copy(str(got), str(GUIDES / g))
print(f'✅ 下载完成，{(time.time() - t0) / 60:.1f} 分钟')


In [ ]:
#@title ③ 启动 ComfyUI 和 H3 Toybox，在下方显示网页
# 在本单元的输出区操作 Toybox。若内嵌页面无法加载，可使用第 ③-b 格。
#@markdown **作品存到Google Drive**：素材库、文件夹、生成的视频、生成纪录都放在「我的Google Drive/H3Toybox」，
#@markdown 断线或不小心关掉运行阶段，重开后再跑一次 ①②③ 东西都还在。第一次会跳出授权窗口，按同意即可。
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown API 密钥也存到Google Drive（下次不用重贴；但Google Drive里会有一份密钥档）
SAVE_API_KEYS = False  #@param {type:"boolean"}
import subprocess, time, urllib.request, json, sys, os, shutil
def wait_http(url, name, log, tries=180):
    for i in range(tries):
        try:
            urllib.request.urlopen(url, timeout=2); print(name, '已启动'); return
        except Exception:
            time.sleep(2)
    print(open(log, encoding='utf-8', errors='ignore').read()[-3000:])
    raise RuntimeError(name + ' 没有启动成功，看上面的纪录')

for proc in ('server', 'toybox'):
    if proc in globals() and globals()[proc].poll() is None:
        globals()[proc].terminate(); globals()[proc].wait()
# 作品放哪里：Google Drive（授权成功）或这台运行阶段的暂存空间
DATA = Path('/content/h3toybox_data')
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA = Path('/content/drive/MyDrive/H3Toybox')
    except Exception as e:
        print('⚠️ 没有挂上Google Drive（', e, '）：这次的作品只存在暂存空间，关掉运行阶段就会消失')
DATA.mkdir(parents=True, exist_ok=True)
# ComfyUI 的输出文件夹直接设在 DATA/output。不能用链接把 output/H3Toybox 接过去：
# ComfyUI 存盘与 /view 都会解开链接检查真实位置，指到 output 外面就报
# 「Saving image outside the output folder is not allowed」。
output_root = DATA / 'output'
videos, out_dir = output_root / 'H3Toybox', COMFY / 'output' / 'H3Toybox'
videos.mkdir(parents=True, exist_ok=True)
# 旧版留下的东西搬过来，以前生成的视频才看得到：DATA/videos、旧链接指向的文件夹、机器上的 output/H3Toybox
old_dirs = [DATA / 'videos']
if out_dir.is_symlink():
    old_dirs.append(out_dir.resolve())
    out_dir.unlink()
elif out_dir.is_dir():
    old_dirs.append(out_dir)
for old in old_dirs:
    if old.is_dir() and old.resolve() != videos.resolve():
        for f in old.iterdir():
            if f.is_file() and not (videos / f.name).exists():
                shutil.move(str(f), str(videos / f.name))
# ComfyUI 只当引擎：只听 127.0.0.1，不对外打开界面
server = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
                           '--output-directory', str(output_root)],
                          cwd=str(COMFY), stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT)
wait_http('http://127.0.0.1:8188/system_stats', 'ComfyUI', '/content/comfyui.log')
h3_image_ready = all((COMFY / 'models' / f).is_file() for f in (*H3_IMAGE_FILES, SHARED_IMAGE_ENCODER))
if h3_image_ready:
    with urllib.request.urlopen('http://127.0.0.1:8188/object_info/MiniMaxH3ImageConditioning', timeout=10) as response:
        if 'MiniMaxH3ImageConditioning' not in json.load(response):
            raise RuntimeError('ComfyUI 缺少 MiniMaxH3ImageConditioning，请检查第 ① 格更新结果和 comfyui.log')
print('🎬 视频写到：', videos)
# H3 Toybox 的 Colab 设置（保留用户在网页设置页改过的内容，只覆写这台机器的路径）
(COMFY / 'input').mkdir(exist_ok=True)
cfg_path = DATA / 'config.json'
cfg = json.loads(cfg_path.read_text(encoding='utf-8')) if cfg_path.exists() else {}
cfg.update({
    'comfy_url': 'http://127.0.0.1:8188',
    'comfy_input_dir': str(COMFY / 'input'),
    'skills_dir': '/content/t8-enhancer/directional_skills',
    'free_after_rewrite': True,   # Colab 内存比较小：改写完就把 9B 模型卸载
})
cfg.pop('guides', None)
# 🖼 图片分页：Colab 用 P-H3-Image（挂在 PRISM 上）；有没有装，看文件是不是真的在
PH3 = ('loras/p-h3-image_lora_bf16.safetensors', 'text_encoders/p-h3-image_text_encoder_int8_convrot.safetensors',
       'vae/p-h3-image_vae_int8_convrot.safetensors')
# 🤖 短片 Agent：Colab 上没有本地大模型，默认改走 Gemini（密钥在网页 ⚙ 设置填）
cfg['agent'] = dict(cfg.get('agent') or {}, provider=(cfg.get('agent') or {}).get('provider', 'gemini'))
cfg['image'] = dict(cfg.get('image') or {}, engine='ph3',
                    installed=all((COMFY / 'models' / f).is_file() for f in PH3))
print('🖼 生图：', '已安装' if cfg['image']['installed'] else '没有安装（② 勾 INSTALL_IMAGE 才有）')
cfg['h3_image'] = {'installed': h3_image_ready}
print('H3-Image：', '权重就绪' if cfg['h3_image']['installed'] else '未安装（在第 ② 格勾选 INSTALL_H3_IMAGE）')
cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=1), encoding='utf-8')
secrets = DATA / 'secrets.json' if SAVE_API_KEYS else Path('/content/h3toybox_secrets.json')
toybox = subprocess.Popen([sys.executable, 'server.py'], cwd=str(APP),
                          env=dict(os.environ, H3TOYBOX_PORT='8190', H3TOYBOX_DATA=str(DATA), H3TOYBOX_SECRETS=str(secrets)),
                          stdout=open('/content/toybox.log', 'w'), stderr=subprocess.STDOUT)
wait_http('http://127.0.0.1:8190/api/config', 'H3 Toybox', '/content/toybox.log', tries=30)
print('📁 作品存放位置：', DATA)
from google.colab import output
output.serve_kernel_port_as_iframe(8190, height='900')


In [ ]:
#@title ③-b（备用）通过 Cloudflare 打开 H3 Toybox
# ⚠️ 网址没有密码，拿到的人都能用你的 GPU —— 不要贴到公开的地方。
ENABLE_TUNNEL = False  #@param {type:"boolean"}
if not ENABLE_TUNNEL:
    print('Cloudflare 通道未开启。请在第 ③ 格的输出区使用 Toybox。')
else:
    import subprocess, re, time
    if not Path('/content/cloudflared').exists():
        subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
        subprocess.run(['chmod', '+x', '/content/cloudflared'], check=True)
    # 通道的输出写进文件，不要接 PIPE：PIPE 没人读会塞满，通道就卡住
    if 'tun' in globals() and tun.poll() is None:
        tun.terminate(); tun.wait()
    open('/content/cloudflared.log', 'w').close()
    tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8190'],
                           stdout=open('/content/cloudflared.log', 'a'), stderr=subprocess.STDOUT)
    url = None
    for i in range(60):
        time.sleep(1)
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
        if m:
            url = m.group(0); break
    print('👉 H3 Toybox：', url or '（没拿到网址，看 /content/cloudflared.log）')


In [ ]:
#@title 🔍 出问题时跑这格：看 GPU、ComfyUI 与 H3 Toybox 的纪录
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)
log = open('/content/comfyui.log', encoding='utf-8', errors='ignore').read()
keys = ('error', 'Error', 'Traceback', 'out of memory', 'SelfLift timing] high_resolution total', 'Prompt executed', 'IMPORT FAILED')
print('\n'.join(l for l in log.splitlines() if any(k in l for k in keys))[-4000:])
print('---- H3 Toybox ----')
print(open('/content/toybox.log', encoding='utf-8', errors='ignore').read()[-2000:])
